# Actividad 2: Modelado del dominio de problemas clasicos de planificación, planificación clasica

**Nombre:** Jesus Manuel Ruiz Fuentes  
**No. de cuenta:** 195116
**Carrera:** Ingenieria en Sistemas Computacionales  
**Asignatura:** Agentes Inteligentes                
**Fecha:** 26 de agosto de 2026

### Modelado del problema de la llanta de repuesto

El objetivo de esta actividad es implementar una solución mediante
planificación clásica utilizando un modelo basado en estados y acciones.

El problema consiste en representar la situación de un automóvil que
tiene una llanta desinflada y cuenta con una llanta de repuesto y las
herramientas necesarias para realizar el cambio.

Para resolver el problema se definirán:

- Un estado inicial.
- Un estado objetivo.
- Un conjunto de acciones.
- Las precondiciones positivas y negativas de cada acción.
- Los efectos de adición (ADD).
- Los efectos de eliminación (DEL).

Posteriormente se comprobará la aplicabilidad de cada acción y se
realizarán las transiciones entre estados hasta alcanzar el objetivo.

El siguiente ejemplo busca ilustrar como dentro de un espacio de estados, un agente busca soluciones aplicando las tareas basicas de planeación.

## 1. Representación del problema

En planificación clásica, un estado representa las condiciones verdaderas del mundo en un momento determinado.

Una acción puede ejecutarse solamente cuando se cumplen sus precondiciones.

Cuando una acción se ejecuta, se obtiene un nuevo estado mediante:

**Resultado(s, a) = (s - DEL(a)) ∪ ADD(a)**

Donde:

- `s` representa el estado actual.
- `DEL(a)` representa los hechos que dejan de ser verdaderos.
- `ADD(a)` representa los hechos que pasan a ser verdaderos.

In [25]:
class AccionPDDL:

    # Definición del esquema de acciones en el marco de PDDL
    def __init__(self, nombre, precondiciones_pos, precondiciones_neg, efectos_a, efectos_d):

        self.nombre = nombre
        self.precondiciones_positivas = set(precondiciones_pos)
        self.precondiciones_negativas = set(precondiciones_neg)
        self.efectos_add = set(efectos_a)
        self.efectos_delete = set(efectos_d)

    # Verifica si la acción es aplicable en un estado dado
    def es_aplicable(self, estado):

        cond_pos_ok = self.precondiciones_positivas.issubset(estado)
        cond_neg_ok = self.precondiciones_negativas.isdisjoint(estado)

        return cond_pos_ok and cond_neg_ok

    # Calcula el estado resultante de aplicar la acción
    def aplicar(self, estado):

        if not self.es_aplicable(estado):
            raise Exception(
                f"Error: la acción {self.nombre} no es aplicable "
                f"en el estado actual"
            )

        estado_siguiente = estado.copy()

        # Eliminar efectos de DEL
        estado_siguiente.difference_update(self.efectos_delete)

        # Agregar efectos de ADD
        estado_siguiente.update(self.efectos_add)

        return estado_siguiente

## 2. Estado inicial

El estado inicial representa la situación del automóvil antes de comenzar el procedimiento.

Se considera que:
- La llanta desinflada está instalada en el automóvil.
- La llanta de repuesto está almacenada en la cajuela.
- Las herramientas están almacenadas en la cajuela.
- El automóvil se encuentra disponible para realizar el cambio.

In [26]:
# Estado inicial
estado_inicial = {
    "llanta_desinflada_en_auto",
    "llanta_repuesto_en_cajuela",
    "herramientas_en_cajuela",
    "auto_en_buen_estado"
}

## 3. Estado objetivo

El objetivo de la planificación es conseguir que la llanta de repuesto
quede instalada en el automóvil.

Por lo tanto, el estado objetivo es:

In [27]:
objetivo = {
    "llanta_repuesto_instalada"
}

## 4. Catálogo de acciones

Para alcanzar el objetivo se establece una secuencia de cuatro acciones:

1. Tomar las herramientas.
2. Desmontar la llanta desinflada.
3. Tomar la llanta de repuesto.
4. Instalar la llanta de repuesto.

Cada acción contiene precondiciones positivas y negativas, así como
efectos ADD y DEL.

In [28]:
tomar_herramientas = AccionPDDL(
    nombre="tomar_herramientas",

    precondiciones_pos={
        "herramientas_en_cajuela"
    },

    precondiciones_neg={
        "herramientas_en_mano"
    },

    efectos_a={
        "herramientas_en_mano"
    },

    efectos_d={
        "herramientas_en_cajuela"
    }
)



desmontar_llanta = AccionPDDL(
    nombre="desmontar_llanta",

    precondiciones_pos={
        "herramientas_en_mano",
        "llanta_desinflada_en_auto"
    },

    precondiciones_neg={
        "llanta_desmontada"
    },

    efectos_a={
        "llanta_desmontada"
    },

    efectos_d={
        "llanta_desinflada_en_auto"
    }
)



tomar_repuesto = AccionPDDL(
    nombre="tomar_llanta_repuesto",

    precondiciones_pos={
        "llanta_repuesto_en_cajuela",
        "llanta_desmontada"
    },

    precondiciones_neg={
        "llanta_repuesto_en_mano"
    },

    efectos_a={
        "llanta_repuesto_en_mano"
    },

    efectos_d={
        "llanta_repuesto_en_cajuela"
    }
)



instalar_llanta = AccionPDDL(
    nombre="instalar_llanta_repuesto",

    precondiciones_pos={
        "llanta_desmontada",
        "llanta_repuesto_en_mano"
    },

    precondiciones_neg={
        "llanta_repuesto_instalada"
    },

    efectos_a={
        "llanta_repuesto_instalada"
    },

    efectos_d={
        "llanta_desmontada",
        "llanta_repuesto_en_mano"
    }
)

In [29]:
estado = estado_inicial.copy()

for accion in [
    tomar_herramientas,
    desmontar_llanta,
    tomar_repuesto,
    instalar_llanta
]:

    print(f"\nAcción: {accion.nombre}")

    if accion.es_aplicable(estado):
        estado = accion.aplicar(estado)
        print("Acción aplicable ✓")
        print("Nuevo estado:", estado)
    else:
        print("Acción NO aplicable ✗")


Acción: tomar_herramientas
Acción aplicable ✓
Nuevo estado: {'auto_en_buen_estado', 'llanta_repuesto_en_cajuela', 'llanta_desinflada_en_auto', 'herramientas_en_mano'}

Acción: desmontar_llanta
Acción aplicable ✓
Nuevo estado: {'llanta_desmontada', 'llanta_repuesto_en_cajuela', 'auto_en_buen_estado', 'herramientas_en_mano'}

Acción: tomar_llanta_repuesto
Acción aplicable ✓
Nuevo estado: {'herramientas_en_mano', 'auto_en_buen_estado', 'llanta_desmontada', 'llanta_repuesto_en_mano'}

Acción: instalar_llanta_repuesto
Acción aplicable ✓
Nuevo estado: {'auto_en_buen_estado', 'herramientas_en_mano', 'llanta_repuesto_instalada'}


In [30]:
if objetivo.issubset(estado):
    print("\n OBJETIVO ALCANZADO")
else:
    print("\n OBJETIVO NO ALCANZADO")


 OBJETIVO ALCANZADO
